## Импорты и настройка

In [1]:
from datasets import load_dataset
from data_structures import Example
from hierarchical_optimizer import HierarchicalOptimizer

/Users/kateaver/idea1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/kateaver/idea1/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/kateav

### Функции для подготовки данных

In [2]:
def squad_v2_to_examples(data):
    examples = []
    for item in data:
        input_text = (f'# Question: \n {item["question"]} \n'
                      f'# Context: \n {item["context"]} \n')
        expected_output = item['answers']['text'][0] if item['answers']['text'] else 'No answer'
        examples.append(Example(input_text=input_text, expected_output=expected_output))
    return examples

def get_squad_v2_data(train_num: int, val_num: int, test_num: int):
    ds_train = load_dataset('rajpurkar/squad_v2', split='train')
    ds_test = load_dataset('rajpurkar/squad_v2', split='validation')

    split = ds_train.train_test_split(test_size=0.1, seed=42)
    ds_train = split['train']
    ds_val = split['test']

    ds_train = ds_train.shuffle()
    ds_val = ds_val.shuffle()
    ds_test = ds_test.shuffle()

    train_split = ds_train.select(range(train_num))
    val_split = ds_val.select(range(val_num))
    test_split = ds_test.select(range(test_num))

    train_examples = squad_v2_to_examples(train_split)
    validation_examples = squad_v2_to_examples(val_split)
    test_examples = squad_v2_to_examples(test_split)

    return train_examples, validation_examples, test_examples

def data_fabric(dataset: str = 'squad_v2', train_num: int = 150, val_num: int = 100, test_num: int = 300):
    squad_v2_initial_prompt = """Answer the question based on the context. If there is no answer in the context then just return 'No answer'."""

    train_examples, validation_examples, test_examples = get_squad_v2_data(train_num, val_num, test_num)
    initial_prompt = squad_v2_initial_prompt
    
    return train_examples, validation_examples, test_examples, initial_prompt

## Подготовка датасета
Создаем простой датасет для демонстрации (задача классификации тональности)

In [3]:
LABEL_MAP = {0: "truth", 1: "lie"}

train_examples, validation_examples, test_examples, initial_prompt = data_fabric('squad_v2')

print("Dataset prepared:")
print(f"  Train: {len(train_examples)} examples")
print(f"  Validation: {len(validation_examples)} examples")
print(f"  Test: {len(test_examples)} examples")

Dataset prepared:
  Train: 150 examples
  Validation: 100 examples
  Test: 300 examples


## Создание начального промпта

In [4]:
print("Initial prompt:")
print("-" * 60)
print(initial_prompt)
print("-" * 60)

Initial prompt:
------------------------------------------------------------
Answer the question based on the context. If there is no answer in the context then just return 'No answer'.
------------------------------------------------------------


## Инициализация оптимизатора

In [5]:
optimizer = HierarchicalOptimizer()

## Запуск оптимизации

In [6]:
best_node = optimizer.optimize(
    initial_prompt=initial_prompt,
    train_examples=train_examples,
    validation_examples=validation_examples,
    test_examples=test_examples,
    save_dir="./optimization_results",
)

Evaluating initial prompt...
[diag] initial node: prompt_id=7c8e9ca8bec6 len=108 chars
[diag] Prompt text: Answer the question based on the context. If there is no answer in the context then just return 'No answer'.
[diag] evaluate_node: node_id=13cd5e88-fea3-4a15-a6a3-837b490cac42 gen=0 source=initial split=validation stage=1 examples=100 seed_offset=0
[diag] evaluate_prompt: execute=True sample=True stage=1 incoming=100 will_use=100 weights={accuracy:0.3, token_f1:0.2} llm_calls_at_start=0
[diag] execute_prompt_batch: total=100 batch_size=50 n_batches=2 llm_calls_before=0
[diag]   batch 1/2: OK (50/100 done, llm_calls=1)
[diag]   batch 2/2: OK (100/100 done, llm_calls=2)
[diag] execute_prompt_batch done: llm_calls_delta=2 total_calls=2
[diag] evaluate_prompt execution done: llm_calls_for_execution=2
[diag]   example[0] expected='about thirty years of age' actual='About thirty years of age'
[diag]   example[1] expected='February 1848' actual='February 1848'
[diag]   example[2] expecte

## Анализ результатов

In [7]:
report = optimizer.get_optimization_report()
print('Optimization generations summary:')
for entry in report['optimization_log']:
    print(f"  Generation {entry['generation']}: time {entry['time']:.2f}s, best_score {entry['best_score']:.3f}")

local_stats = report['component_statistics']['local_optimizer']
print('Local optimizer summary:')
print(f"  Total iterations recorded: {local_stats.get('total_iterations', 0)}")
avg_it = local_stats.get('avg_iteration_time')
if avg_it is not None:
    print(f"  Avg iteration time: {avg_it:.2f}s")
else:
    print('  Avg iteration time: N/A')
print(f"  Total LLM calls attributed to local iterations: {local_stats.get('total_llm_calls_by_local', 0)}")
print('Per-iteration breakdown:')
for s in local_stats.get('iteration_stats', []):
    print(f"  Iter {s['iteration']}: time {s['time']:.2f}s, llm_calls {s['llm_calls']}")

Optimization generations summary:
  Generation 1: time 493.39s, best_score 0.450
  Generation 2: time 1549.08s, best_score 0.505
  Generation 3: time 370.38s, best_score 0.528
  Generation 4: time 475.75s, best_score 0.528
  Generation 5: time 332.89s, best_score 0.528
Local optimizer summary:
  Total iterations recorded: 14
  Avg iteration time: 200.54s
  Total LLM calls attributed to local iterations: 381
Per-iteration breakdown:
  Iter 1: time 182.08s, llm_calls 19
  Iter 2: time 309.91s, llm_calls 37
  Iter 1: time 175.50s, llm_calls 18
  Iter 2: time 183.91s, llm_calls 19
  Iter 1: time 173.15s, llm_calls 17
  Iter 2: time 301.04s, llm_calls 29
  Iter 1: time 189.56s, llm_calls 18
  Iter 2: time 223.13s, llm_calls 67
  Iter 1: time 206.59s, llm_calls 66
  Iter 2: time 162.02s, llm_calls 15
  Iter 1: time 186.68s, llm_calls 17
  Iter 2: time 181.03s, llm_calls 28
  Iter 1: time 161.56s, llm_calls 16
  Iter 2: time 171.33s, llm_calls 15


In [8]:
print("BEST PROMPT FOUND:")
print("=" * 80)
print(best_node.prompt_text)
print("=" * 80)
print(f"Score: {best_node.metrics.composite_score():.3f}")
print(f"Generation: {best_node.generation}")
print(f"Source: {best_node.source.value}")

BEST PROMPT FOUND:
Analyze the context provided to determine if the answer is present. If there are multiple possible answers, prioritize the most specific one. Provide the answer concisely in the correct format. If the necessary information is missing or ambiguous, respond with 'No answer' and briefly explain why the question cannot be answered.
Score: 0.528
Generation: 4
Source: local


In [11]:
print(optimizer.visualize_optimization_trajectory())


OPTIMIZATION TRAJECTORY

Generation | Best Score | Overall Best | Improvement
------------------------------------------------------------
   1       | 0.450      | 0.450       | +0.145 ██████████████████████
   2       | 0.505      | 0.505       | +0.056 █████████████████████████
   3       | 0.528      | 0.528       | +0.023 ██████████████████████████
   4       | 0.528      | 0.528       | +0.000 ██████████████████████████
   5       | 0.528      | 0.528       | +0.000 ██████████████████████████




In [12]:
ыreport = optimizer.get_optimization_report()

print("OPTIMIZATION REPORT:")
print("Overall Statistics:")
print(f"   Total time: {report['optimization_info']['total_time_seconds']:.2f}s")
print(f"   Generations: {report['optimization_info']['generations']}")
print(f"   Total nodes explored: {report['component_statistics']['history']['total_nodes']}")

print("Component Statistics:")
print(f"   Local optimizer iterations: {report['component_statistics']['local_optimizer']['total_iterations']}")
print(f"   Local improvements: {report['component_statistics']['local_optimizer']['improvements_count']}")
print(f"   Global optimizer steps: {report['component_statistics']['global_optimizer']['total_global_steps']}")
print(f"   Successful global changes: {report['component_statistics']['global_optimizer']['successful_global_changes']}")

print("Best Global Strategies:")
for i, strategy in enumerate(report['best_global_strategies'][:3], 1):
    print(f"   {i}. Score {strategy['score']:.3f}")
    print(f"      {strategy['strategy']['description'][:70]}...")


OPTIMIZATION REPORT:
Overall Statistics:
   Total time: 3247.86s
   Generations: 5
   Total nodes explored: 49
Component Statistics:
   Local optimizer iterations: 14
   Local improvements: 5
   Global optimizer steps: 2
   Successful global changes: 1
Best Global Strategies:
   1. Score 0.469
      meta-optimizer...
   2. Score 0.441
      crossover...
   3. Score 0.388
      meta-optimizer...


In [13]:
lineage = optimizer.history.get_lineage(best_node.id)

print("EVOLUTION OF BEST PROMPT:")
print("="*80)

for i, node in enumerate(lineage):
    print(f"Step {i}: Generation {node.generation}, Source: {node.source.value}")
    if node.is_evaluated:
        print(f"  Score: {node.metrics.composite_score():.3f}")

    if node.operations:
        print(f"  Operations:")
        for op in node.operations:
            print(f"    - {op.description}...")

    if i < len(lineage) - 1:  
        print("  ↓")


EVOLUTION OF BEST PROMPT:
Step 0: Generation 0, Source: initial
  Score: 0.304
  ↓
Step 1: Generation 1, Source: local
  Score: 0.384
  Operations:
    - Introduced a decision rule for cases where the answer cannot be found in the context....
  ↓
Step 2: Generation 2, Source: local
  Score: 0.450
  Operations:
    - Added a requirement for the model to provide reasoning along with the answer to ensure accuracy....
  ↓
Step 3: Generation 2, Source: global
  Score: 0.469
  Operations:
    - Meta-optimizer (gen 2)...
  ↓
Step 4: Generation 3, Source: local
  Score: 0.505
  Operations:
    - Included an explicit output format constraint to ensure the model's responses are properly structured....
  ↓
Step 5: Generation 4, Source: local
  Score: 0.528
  Operations:
    - Adding a disambiguation rule to guide the model in selecting the most specific answer when multiple options are available....


In [14]:
print("FINAL SUMMARY")
print("="*80)

history_stats = optimizer.history.get_statistics()
local_stats = optimizer.local_optimizer.get_statistics()
global_stats = optimizer.global_optimizer.get_statistics()

print("Overall Statistics:")
print(f"  Total nodes explored: {history_stats['total_nodes']}")
print(f"  Evaluations performed: {history_stats['evaluated_nodes']}")
print(f"  Generations completed: {history_stats['max_generation']}")
print(f"  Best score achieved: {history_stats['best_score']:.3f}")
print(f"  Average score: {history_stats['avg_score']:.3f}")

print("Local Optimization:")
print(f"  Total iterations: {local_stats['total_iterations']}")
print(f"  Improvements found: {local_stats['improvements_count']}")
print(f"  Success rate: {local_stats['improvement_rate']:.1%}")

print("Global Optimization:")
print(f"  Total global steps: {global_stats['total_global_steps']}")
print(f"  Candidates generated: {global_stats['total_candidates_generated']}")
print(f"  Successful changes: {global_stats['successful_global_changes']}")
print(f"  Success rate: {global_stats['success_rate']:.1%}")

print("Optimization complete!")
print(f"   Results saved to: ./optimization_results/")

FINAL SUMMARY
Overall Statistics:
  Total nodes explored: 49
  Evaluations performed: 49
  Generations completed: 5
  Best score achieved: 0.528
  Average score: 0.411
Local Optimization:
  Total iterations: 14
  Improvements found: 5
  Success rate: 35.7%
Global Optimization:
  Total global steps: 2
  Candidates generated: 12
  Successful changes: 1
  Success rate: 8.3%
Optimization complete!
   Results saved to: ./optimization_results/
